In [ ]:
import pandas as pd

DATA_PATH = "path_to_dataset"

df = pd.read_json(
    DATA_PATH,
    lines=True
)

df.head()

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

# shuffle
dataset = dataset.shuffle(seed=42)

# 80% train
train_test = dataset.train_test_split(test_size=0.2, seed=42)

# 10% val + 10% test
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

train_data = train_test["train"]
val_data   = val_test["train"]
test_data  = val_test["test"]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

model.config.use_cache = False

In [ ]:
def preprocess(example):
    inputs = tokenizer(
        example["source"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        example["target"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

train_data = train_data.map(preprocess)
val_data   = val_data.map(preprocess)
test_data  = test_data.map(preprocess)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./talabasa_model",

    # ===== EPOCHS =====
    num_train_epochs=3,

    # ===== BATCH =====
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    # ===== SPEED / MEMORY =====
    fp16=True,
    gradient_checkpointing=True,

    # ===== OPTIMIZER =====
    learning_rate=2e-5,
    warmup_steps=1000,
    weight_decay=0.01,

    # ===== EVALUATION =====
    eval_strategy="steps",
    eval_steps=500,

    # ===== CHECKPOINTS =====
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,

    # ===== BEST MODEL =====
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ===== LOGGING =====
    logging_steps=50,
    dataloader_num_workers=0,

    report_to="none"
)

In [ ]:
from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2  # stops if there is no improvement for 2 evals
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    callbacks=[early_stopping]
)

trainer.train()